<a href="https://colab.research.google.com/github/b03343028-sketch/DecodeLab--internship-bot/blob/main/Copy_of_recomandation_system_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
=========================================================
AI RECOMMENDATION SYSTEM
Part 1 (1-100)
=========================================================
"""

from __future__ import annotations
import json
import math
import os
import hashlib
from dataclasses import dataclass, field
from typing import Dict, List
from datetime import datetime

# =========================================================
# CONSTANTS
# =========================================================

USER_FILE = "users.json"
CATALOG_FILE = "catalog.json"

LEARNING_RATE = 0.15
TOP_RESULTS = 5

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()


def load_json(filename, default):

    if os.path.exists(filename):

        try:
            with open(filename, "r", encoding="utf-8") as file:
                return json.load(file)

        except:
            return default

    return default


def save_json(filename, data):

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)


# =========================================================
# DATA CLASSES
# =========================================================

@dataclass
class Item:

    item_id: int
    name: str
    category: str
    tags: Dict[str, float]
    description: str = ""
    rating: float = 0
    votes: int = 0

    def show(self):

        print("-" * 60)
        print("ID          :", self.item_id)
        print("Name        :", self.name)
        print("Category    :", self.category)
        print("Description :", self.description)
        print("Rating      :", round(self.rating, 2))
        print("Tags        :", ", ".join(self.tags.keys()))
        print("-" * 60)


@dataclass
class UserProfile:

    username: str
    interests: Dict[str, float] = field(default_factory=dict)
    likes: List[int] = field(default_factory=list)
    dislikes: List[int] = field(default_factory=list)
    favorites: List[int] = field(default_factory=list)
    history: List[dict] = field(default_factory=list)

    def add_interest(self, tag):

        self.interests[tag] = self.interests.get(tag, 0) + LEARNING_RATE

    def remove_interest(self, tag):

        if tag in self.interests:

            self.interests[tag] = max(
                0,
                self.interests[tag] - LEARNING_RATE
            )

    def add_history(self, item):

        self.history.append({

            "item": item,
            "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        })

# =========================================================
# PART 2 (101-200)
# USER MANAGEMENT & CATALOG
# =========================================================

def register():

    users = load_json(USER_FILE, {})

    username = input("Enter Username : ").strip()

    if username in users:
        print("Username already exists.")
        return

    password = input("Enter Password : ")

    users[username] = {
        "password": hash_password(password),
        "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    save_json(USER_FILE, users)

    print("Registration Successful.")


def login():

    users = load_json(USER_FILE, {})

    username = input("Username : ").strip()
    password = input("Password : ")

    if username not in users:
        print("User not found.")
        return None

    if users[username]["password"] != hash_password(password):
        print("Incorrect Password.")
        return None

    print("Login Successful.")
    return username


# =========================================================
# SAMPLE CATALOG
# =========================================================

def build_catalog():

    return [

        Item(
            1,
            "Inception",
            "Movie",
            {"sci-fi":1.0,"action":0.8,"thriller":0.7},
            "Dream based science fiction movie."
        ),

        Item(
            2,
            "Interstellar",
            "Movie",
            {"space":1.0,"science":0.9,"drama":0.7},
            "Journey through space."
        ),

        Item(
            3,
            "Clean Code",
            "Book",
            {"programming":1.0,"technology":0.8},
            "Programming best practices."
        ),

        Item(
            4,
            "Python for Data Science",
            "Course",
            {"python":1.0,"data":1.0,"ai":0.8},
            "Complete Python course."
        ),

        Item(
            5,
            "Machine Learning A-Z",
            "Course",
            {"ai":1.0,"python":0.8,"data":0.9},
            "Machine Learning course."
        )

    ]


def get_item(catalog, item_id):

    for item in catalog:

        if item.item_id == item_id:
            return item

    return None


# =========================================================
# SIMILARITY FUNCTIONS
# =========================================================

def jaccard(profile, item):

    a = set(profile.interests.keys())
    b = set(item.tags.keys())

    if not a or not b:
        return 0

    return len(a & b) / len(a | b)


def cosine(profile, item):

    common = set(profile.interests.keys()) & set(item.tags.keys())

    if not common:
        return 0

    dot = sum(profile.interests[t] * item.tags[t] for t in common)

    mag1 = math.sqrt(sum(v*v for v in profile.interests.values()))
    mag2 = math.sqrt(sum(v*v for v in item.tags.values()))

    if mag1 == 0 or mag2 == 0:
        return 0

    return dot / (mag1 * mag2)

# =========================================================
# PART 3 (201-300)
# RECOMMENDATION ENGINE
# =========================================================

def hybrid_score(profile, item):

    j = jaccard(profile, item)
    c = cosine(profile, item)

    return (0.4 * j) + (0.6 * c)


def recommend(profile, catalog):

    recommendations = []

    for item in catalog:

        if item.item_id in profile.dislikes:
            continue

        score = hybrid_score(profile, item)

        if score > 0:

            recommendations.append((item, score))

    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations[:TOP_RESULTS]


def show_recommendations(profile, catalog):

    results = recommend(profile, catalog)

    print("\nRecommended Items")
    print("=" * 60)

    if len(results) == 0:

        print("No recommendations available.")
        return

    rank = 1

    for item, score in results:

        print(rank)
        print("ID       :", item.item_id)
        print("Name     :", item.name)
        print("Category :", item.category)
        print("Match    :", round(score * 100, 2), "%")
        print("-" * 60)

        rank += 1


# =========================================================
# SEARCH
# =========================================================

def search_item(catalog):

    keyword = input("Search : ").lower()

    found = False

    for item in catalog:

        if keyword in item.name.lower():

            item.show()

            found = True

    if not found:
        print("No Item Found.")


def browse_catalog(catalog):

    print("\nCatalog")
    print("=" * 60)

    for item in catalog:

        print(
            item.item_id,
            item.name,
            "(" + item.category + ")"
        )


# =========================================================
# FEEDBACK
# =========================================================

def like_item(profile, catalog):

    item_ids_input = input("Item ID(s) (comma-separated for multiple): ")
    item_ids_str = [id_str.strip() for id_str in item_ids_input.split(',')]

    for id_str in item_ids_str:
        try:
            item_id = int(id_str)
        except ValueError:
            print(f"Invalid Item ID: {id_str}. Skipping.")
            continue

        item = get_item(catalog, item_id)

        if item is None:

            print(f"Item with ID {item_id} not found. Skipping.")
            continue

        if item_id not in profile.likes:

            profile.likes.append(item_id)

        for tag, weight in item.tags.items():

            profile.interests[tag] = (
                profile.interests.get(tag, 0)
                + weight * LEARNING_RATE
            )

        profile.add_history(item.name)

        print(f"Item {item.name} Liked Successfully.")


def dislike_item(profile, catalog):

    item_ids_input = input("Item ID(s) (comma-separated for multiple): ")
    item_ids_str = [id_str.strip() for id_str in item_ids_input.split(',')]

    for id_str in item_ids_str:
        try:
            item_id = int(id_str)
        except ValueError:
            print(f"Invalid Item ID: {id_str}. Skipping.")
            continue

        item = get_item(catalog, item_id)

        if item is None:

            print(f"Item with ID {item_id} not found. Skipping.")
            continue

        if item_id not in profile.dislikes:

            profile.dislikes.append(item_id)

        for tag in item.tags:

            profile.interests[tag] = max(
                0,
                profile.interests.get(tag, 0)
                - LEARNING_RATE
            )

        profile.add_history(item.name)

        print(f"Item {item.name} Disliked Successfully.")

def add_favorite(profile, catalog):

    item_ids_input = input("Item ID(s) (comma-separated for multiple): ")
    item_ids_str = [id_str.strip() for id_str in item_ids_input.split(',')]

    for id_str in item_ids_str:
        try:
            item_id = int(id_str)
        except ValueError:
            print(f"Invalid Item ID: {id_str}. Skipping.")
            continue

        item = get_item(catalog, item_id)

        if item is None:
            print(f"Item with ID {item_id} not found. Skipping.")
            continue

        if item_id not in profile.favorites:
            profile.favorites.append(item_id)
            print(f"Item {item.name} Added to Favorites.")
        else:
            print(f"Item {item.name} is already in favorites.")


def view_favorites(profile, catalog):

    print("\nFavorite Items")
    print("=" * 50)

    if len(profile.favorites) == 0:
        print("No Favorites.")
        return

    for item_id in profile.favorites:

        item = get_item(catalog, item_id)

        if item:
            print(item.item_id, "-", item.name)


def view_history(profile):

    print("\nHistory")
    print("=" * 50)

    if len(profile.history) == 0:
        print("No History.")
        return

    for record in profile.history:
        print(record["time"], "-", record["item"])


def rate_item(catalog):

    item_ids_input = input("Item ID(s) to rate (comma-separated for multiple): ")
    item_ids_str = [id_str.strip() for id_str in item_ids_input.split(',')]

    for id_str in item_ids_str:
        try:
            item_id = int(id_str)
        except ValueError:
            print(f"Invalid Item ID: {id_str}. Skipping rating.")
            continue

        item = get_item(catalog, item_id)

        if item is None:
            print(f"Item with ID {item_id} not found. Skipping rating.")
            continue

        try:
            rating = float(input(f"Rating for {item.name} (1-5): "))
        except ValueError:
            print("Invalid Rating. Skipping.")
            continue

        if rating < 1 or rating > 5:
            print("Rating must be between 1 and 5. Skipping.")
            continue

        total = (item.rating * item.votes) + rating

        item.votes += 1

        item.rating = total / item.votes

        print(f"Rating for {item.name} Saved.")


def menu(profile, catalog):

    while True:

        print("\n========== AI RECOMMENDATION SYSTEM ==========")
        print("1. Browse Catalog")
        print("2. Search Item")
        print("3. Show Recommendations")
        print("4. Like Item")
        print("5. Dislike Item")
        print("6. Add Favorite")
        print("7. View Favorites")
        print("8. Rate Item")
        print("9. View History")
        print("10. Logout")

        choice = input("Choice : ")

        if choice == "1":
            browse_catalog(catalog)

        elif choice == "2":
            search_item(catalog)

        elif choice == "3":
            show_recommendations(profile, catalog)

        elif choice == "4":
            like_item(profile, catalog)

        elif choice == "5":
            dislike_item(profile, catalog)

        elif choice == "6":
            add_favorite(profile, catalog)

        elif choice == "7":
            view_favorites(profile, catalog)

        elif choice == "8":
            rate_item(catalog)

        elif choice == "9":
            view_history(profile)

        elif choice == "10":
            print("Logged Out.")
            break

        else:
            print("Invalid Choice.")


def main():

    catalog = build_catalog()

    while True:

        print("\n========== MAIN MENU ==========")
        print("1. Register")
        print("2. Login")
        print("3. Exit")

        option = input("Choice : ")

        if option == "1":

            register()

        elif option == "2":

            username = login()

            if username:

                profile = UserProfile(username)

                menu(profile, catalog)

        elif option == "3":

            print("Thank You for using AI Recommendation System.")
            break

        else:

            print("Invalid Option.")


if __name__ == "__main__":

    main()



========== MAIN MENU ==========
1. Register
2. Login
3. Exit
Choice : 1
Enter Username : tahleel
Username already exists.

========== MAIN MENU ==========
1. Register
2. Login
3. Exit
Choice : ayesha
Invalid Option.

========== MAIN MENU ==========
1. Register
2. Login
3. Exit
Choice : 1
Enter Username : ayesha
Enter Password : 6789
Registration Successful.

========== MAIN MENU ==========
1. Register
2. Login
3. Exit
Choice : 2
Username : ayesha
Password : 6789
Login Successful.

========== AI RECOMMENDATION SYSTEM ==========
1. Browse Catalog
2. Search Item
3. Show Recommendations
4. Like Item
5. Dislike Item
6. Add Favorite
7. View Favorites
8. Rate Item
9. View History
10. Logout
Choice : 4
Item ID : 4
Liked Successfully.

========== AI RECOMMENDATION SYSTEM ==========
1. Browse Catalog
2. Search Item
3. Show Recommendations
4. Like Item
5. Dislike Item
6. Add Favorite
7. View Favorites
8. Rate Item
9. View History
10. Logout
Choice : 3

Recommended Items
1
ID       : 4
Name     :